In [38]:
import pandas as pd

stud_info = pd.read_csv("tables/student_info.csv")
stud_info

,student_id,date_registered
0,258798,2022-01-01
1,258799,2022-01-01
2,258800,2022-01-01
3,258801,2022-01-01
4,258802,2022-01-01
...,...,...
37485,297050,2022-10-31
37486,297052,2022-10-31
37487,297053,2022-10-31
37488,297054,2022-10-31


In [39]:
subs = pd.read_csv("tables/student_purchases.csv")
subs

,purchase_id,student_id,purchase_type,date_purchased,date_refunded
0,15781,258800,2,2022-01-01,\N
1,15786,258803,2,2022-01-01,\N
2,15808,258862,2,2022-01-01,\N
3,15809,258865,2,2022-01-01,\N
4,15811,258878,2,2022-01-01,\N
...,...,...,...,...,...
3325,23345,296965,1,2022-10-31,\N
3326,23346,293600,2,2022-10-31,\N
3327,23347,296613,0,2022-10-31,\N
3328,23348,282158,1,2022-10-31,2022-11-03


In [40]:
df = stud_info
df["f2p"] = df.student_id.isin(subs.student_id).astype(int)
df

,student_id,date_registered,f2p
0,258798,2022-01-01,0
1,258799,2022-01-01,0
2,258800,2022-01-01,1
3,258801,2022-01-01,0
4,258802,2022-01-01,0
...,...,...,...
37485,297050,2022-10-31,0
37486,297052,2022-10-31,0
37487,297053,2022-10-31,0
37488,297054,2022-10-31,0


In [41]:
stud_learn = pd.read_csv("tables/student_learning.csv")
stud_learn

,student_id,course_id,minutes_watched,date_watched
0,258940,2,2.12,2022-01-02
1,258893,2,0.65,2022-01-02
2,258904,2,13.08,2022-01-02
3,259041,2,0.18,2022-01-02
4,258904,2,36.33,2022-01-03
...,...,...,...,...
85940,293600,67,159.53,2022-10-31
85941,294296,67,36.38,2022-10-31
85942,268331,67,99.85,2022-10-31
85943,262580,67,8.22,2022-10-31


In [43]:
stud_learn.date_watched = pd.to_datetime(stud_learn.date_watched)

In [18]:
stud_learn[stud_learn.student_id == 258800].minutes_watched.sum()

np.float64(531.02)

In [21]:
subs[subs.student_id == 258803].date_purchased.min()

'2022-01-01'

In [44]:
import numpy as np

total_mins = []
for i, f2p in zip(df.student_id, df.f2p):
    if f2p == 0:
        total_mins.append(stud_learn[stud_learn.student_id == i].minutes_watched.sum())
    else:
        first_sub = subs[subs.student_id == i].date_purchased.min()
        total_mins.append(stud_learn[(stud_learn.student_id == i) & (stud_learn.date_watched <= np.datetime64(first_sub))].minutes_watched.sum())

df["total_minutes_watched"] = total_mins
df

,student_id,date_registered,f2p,total_minutes_watched
0,258798,2022-01-01,0,0.28
1,258799,2022-01-01,0,0.00
2,258800,2022-01-01,1,0.00
3,258801,2022-01-01,0,0.00
4,258802,2022-01-01,0,0.00
...,...,...,...,...
37485,297050,2022-10-31,0,13.17
37486,297052,2022-10-31,0,2.20
37487,297053,2022-10-31,0,0.07
37488,297054,2022-10-31,0,0.00


In [45]:
a = [10,21,30,40,50]
pd.cut(a, bins=[0,20,40,60,80])

[(0, 20], (20, 40], (20, 40], (20, 40], (40, 60]]
Categories (4, interval[int64, right]): [(0, 20] < (20, 40] < (40, 60] < (60, 80]]

In [46]:
df.total_minutes_watched.max()

2697.7699999999995

In [48]:
def bins_generator(maximum):
    i = 0
    while i < maximum:
        yield i
        if i < 30:
            i += 5
        elif i < 120:
            i += 10
        elif i < 480:
            i *= 2
        elif i == 480:
            i = 1000
        elif i < 2000:
            i += 1000
        else:
            yield maximum
            i = maximum

list(bins_generator(df.total_minutes_watched.max()))

[0,
 5,
 10,
 15,
 20,
 25,
 30,
 40,
 50,
 60,
 70,
 80,
 90,
 100,
 110,
 120,
 240,
 480,
 1000,
 2000,
 2697.7699999999995]

In [49]:
pd.cut(df.total_minutes_watched, bins=bins_generator(df.total_minutes_watched.max()))

0          (0.0, 5.0]
1                 NaN
2                 NaN
3                 NaN
4                 NaN
             ...     
37485    (10.0, 15.0]
37486      (0.0, 5.0]
37487      (0.0, 5.0]
37488             NaN
37489      (0.0, 5.0]
Name: total_minutes_watched, Length: 37490, dtype: category
Categories (20, interval[float64, right]): [(0.0, 5.0] < (5.0, 10.0] < (10.0, 15.0] < (15.0, 20.0] ... (240.0, 480.0] < (480.0, 1000.0] < (1000.0, 2000.0] < (2000.0, 2697.77]]

In [50]:
df["buckets"] = pd.cut(df.total_minutes_watched, bins=bins_generator(df.total_minutes_watched.max()))
df

,student_id,date_registered,f2p,total_minutes_watched,buckets
0,258798,2022-01-01,0,0.28,"(0.0, 5.0]"
1,258799,2022-01-01,0,0.00,NaN
2,258800,2022-01-01,1,0.00,NaN
3,258801,2022-01-01,0,0.00,NaN
4,258802,2022-01-01,0,0.00,NaN
...,...,...,...,...,...
37485,297050,2022-10-31,0,13.17,"(10.0, 15.0]"
37486,297052,2022-10-31,0,2.20,"(0.0, 5.0]"
37487,297053,2022-10-31,0,0.07,"(0.0, 5.0]"
37488,297054,2022-10-31,0,0.00,NaN


In [51]:
df.buckets = df.buckets.cat.add_categories([0]).fillna(0)
df

,student_id,date_registered,f2p,total_minutes_watched,buckets
0,258798,2022-01-01,0,0.28,"(0.0, 5.0]"
1,258799,2022-01-01,0,0.00,0
2,258800,2022-01-01,1,0.00,0
3,258801,2022-01-01,0,0.00,0
4,258802,2022-01-01,0,0.00,0
...,...,...,...,...,...
37485,297050,2022-10-31,0,13.17,"(10.0, 15.0]"
37486,297052,2022-10-31,0,2.20,"(0.0, 5.0]"
37487,297053,2022-10-31,0,0.07,"(0.0, 5.0]"
37488,297054,2022-10-31,0,0.00,0


In [52]:
df.buckets.value_counts()

buckets
0                    18771
(0.0, 5.0]            8428
(5.0, 10.0]           2044
(10.0, 15.0]          1202
(40.0, 50.0]          1164
(30.0, 40.0]          1037
(15.0, 20.0]           855
(120.0, 240.0]         594
(20.0, 25.0]           593
(50.0, 60.0]           579
(25.0, 30.0]           525
(60.0, 70.0]           422
(70.0, 80.0]           309
(80.0, 90.0]           227
(90.0, 100.0]          188
(240.0, 480.0]         186
(100.0, 110.0]         163
(110.0, 120.0]         138
(480.0, 1000.0]         49
(1000.0, 2000.0]        13
(2000.0, 2697.77]        3
Name: count, dtype: int64

In [53]:
df.to_csv("data/f2p.csv", index=False)